# Baseline PM2.5 Bogotá — 3 fases de tratamiento de datos faltantes

Este cuaderno construye un **modelo baseline de Random Forest** para pronóstico de PM2.5 a 1 hora,
y lo evalúa bajo **tres estrategias distintas de tratamiento de datos faltantes**, manteniendo
**el mismo modelo, la misma ventana temporal y el mismo conjunto de test** en las tres — así
cualquier diferencia de desempeño se puede atribuir limpiamente a la estrategia de datos, no al modelo.

## Diseño experimental (resumen)

| Fase | Estrategia | Rol |
|---|---|---|
| 0 | Persistencia (`ŷ(t) = y(t-1)`) | Línea base obligatoria — sin esto no sabemos si el RF aporta algo real |
| 3 | FONTIBÓN, solo variables con cobertura nativa suficiente (sin imputar) | Control / piso — RF con datos 100% reales |
| 1 | FONTIBÓN + imputación MissForest (intra-estación) | ¿Ayuda la imputación estadística sola? |
| 2 | FONTIBÓN + MissForest + vecinos vía grafo de distancia | ¿Ayuda además la información espacial? |

## Reglas que se respetan en todo el cuaderno (no negociables)

1. **Split temporal, no aleatorio**: train = 2020-01-01 a 2023-12-31, test = 2024 completo (año-independiente,
   igual que el protocolo del paper de referencia en Beijing).
2. **El imputador (MissForest) se ajusta *solo* con el período de train** y se aplica hacia adelante sobre
   test — nunca se ajusta con el dataset completo (evita fuga de información del futuro).
3. **El escalador, si se usa, también se ajusta solo con train.**
4. **El target de evaluación (PM2.5 de test) es siempre el valor real observado, nunca un valor imputado.**
   Si una hora de test tiene el PM2.5 faltante, esa hora se excluye de la evaluación, no se rellena para evaluarla.
5. Se miden y separan tres tiempos por fase: tiempo de imputación, tiempo de entrenamiento del RF,
   y tiempo de inferencia (esta última es la que se puede comparar directamente contra la columna
   "Execution time" de la Tabla 1 del paper, que mide solo tiempo sobre el set de test).

## Requisitos para correr este cuaderno

Este cuaderno espera encontrar `bogota_calidad_aire_raw.parquet` (el caché generado por el EDA) en la
misma carpeta, o ajustar `RUTA_PARQUET` en la Sección 1. Necesita `pyarrow` o `fastparquet` instalado
para leer el parquet — si no los tenés: `pip install pyarrow`.


---

## Sección 1: Librerías y parámetros globales

**Qué se hace:** se importan las librerías y se fijan en un solo lugar todos los parámetros que
deben mantenerse iguales entre las tres fases (ventana temporal, lookback, split, umbral de
cobertura, etc.), para que ningún parámetro se "deslice" entre fases sin que se note.


In [ ]:
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer  # necesario para habilitar IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style('whitegrid')
print('Librerías cargadas correctamente.')

In [ ]:
# ── Parámetros globales — fijos para las 3 fases ───────────────────────────

RUTA_PARQUET = 'bogota_calidad_aire_raw.parquet'   # ajustar si está en otra carpeta

ESTACION_OBJETIVO = 'FONTIBÓN'

# Ventana temporal común a todo el proyecto (ver EDA, Sección 5.1):
# las 22 estaciones con PM2.5 arrancan el 2020-01-01, por eso se fija esta ventana.
FECHA_INICIO = '2020-01-01'
FECHA_FIN_EXCLUSIVA = '2025-01-01'   # límite exclusivo, cubre hasta 2024-12-31 23:00

# Split año-independiente, igual protocolo que el paper en Beijing (train 4 años, test 1 año)
FECHA_CORTE_TRAIN_TEST = '2024-01-01'   # train: [FECHA_INICIO, FECHA_CORTE); test: [FECHA_CORTE, FECHA_FIN)

LOOKBACK = 24          # ventana de lags, igual que el paper (h1: 1 paso adelante)
VARIABLES_REPLICA = ['PM2.5', 'TAire2', 'TAire10', 'HAire2', 'HAire10',
                      'P', 'DViento', 'VViento', 'PLiquida']

# Umbral de cobertura nativa (Fase 3): variables con menos de este % de horas observadas
# en el período de TRAIN se descartan del feature set en la Fase 3 (control), en vez de imputarse.
UMBRAL_COBERTURA_FASE3 = 0.70

N_ESTIMATORS_RF = 200
RANDOM_STATE = 42

print(f'Estación objetivo: {ESTACION_OBJETIVO}')
print(f'Train: [{FECHA_INICIO}, {FECHA_CORTE_TRAIN_TEST})')
print(f'Test:  [{FECHA_CORTE_TRAIN_TEST}, {FECHA_FIN_EXCLUSIVA})')

---

## Sección 2: Carga y pivote largo → ancho

**Qué se hace:** se carga el parquet crudo (formato largo: una fila por medición), se filtra a las
variables de la réplica, y se define una función reutilizable que pivota **cualquier subconjunto de
estaciones** a formato ancho (una fila por hora, una columna por variable×estación), reindexado a
una grilla horaria completa y continua entre `FECHA_INICIO` y `FECHA_FIN_EXCLUSIVA`.

**Por qué una función reutilizable:** las tres fases y, más adelante, el pooling multi-estación,
necesitan exactamente esta misma transformación. Escribirla una sola vez evita que se cuelen
diferencias de preprocesamiento entre fases (lo que invalidaría la comparación).


In [ ]:
t0 = time.perf_counter()
df_raw = pd.read_parquet(RUTA_PARQUET)
print(f'Parquet cargado: {len(df_raw):,} filas en {time.perf_counter()-t0:.1f}s')

# Estandarización mínima de tipos (igual que en el EDA)
df_raw['med_fecha_inicio'] = pd.to_datetime(df_raw['med_fecha_inicio'])
df_raw['med_concentracion_estandar'] = pd.to_numeric(
    df_raw['med_concentracion_estandar'], errors='coerce'
)

df = df_raw[df_raw['msfl_code'].isin(VARIABLES_REPLICA)].copy()
del df_raw  # liberar memoria, ya no se necesita el crudo completo
print(f'Filas tras filtrar a variables de la réplica: {len(df):,}')

In [ ]:
def pivotear_ancho(df_largo, estaciones, fecha_inicio, fecha_fin_exclusiva):
    \"\"\"
    Convierte el formato largo (fila = medición) a formato ancho, reindexado a
    una grilla horaria continua, para una o varias estaciones.

    Si se pasa más de una estación, las columnas de las estaciones "vecinas"
    (todas menos la primera de la lista) se prefijan con el nombre de la estación,
    para poder usarlas como predictores adicionales sin confundirlas con las
    variables de la estación objetivo.

    Devuelve un DataFrame indexado por fecha-hora, ordenado cronológicamente.
    \"\"\"
    grilla = pd.date_range(fecha_inicio, fecha_fin_exclusiva, freq='h', inclusive='left')

    tablas = []
    for i, est in enumerate(estaciones):
        sub = df_largo[df_largo['nombre_est'] == est].copy()
        sub = sub[(sub['med_fecha_inicio'] >= fecha_inicio) &
                   (sub['med_fecha_inicio'] < fecha_fin_exclusiva)]

        ancha = sub.pivot_table(
            index='med_fecha_inicio', columns='msfl_code',
            values='med_concentracion_estandar', aggfunc='mean'
        )
        ancha = ancha.reindex(grilla)

        if i > 0:
            ancha.columns = [f'{est}__{c}' for c in ancha.columns]

        tablas.append(ancha)

    resultado = pd.concat(tablas, axis=1)
    resultado.index.name = 'fecha_hora'
    return resultado


# Tabla ancha de la estación objetivo sola (se usa en Fases 0, 1 y 3)
df_ancho_fontibon = pivotear_ancho(df, [ESTACION_OBJETIVO], FECHA_INICIO, FECHA_FIN_EXCLUSIVA)
print('Forma de la tabla ancha de FONTIBÓN:', df_ancho_fontibon.shape)
df_ancho_fontibon.head()

---

## Sección 3: Construcción de ventanas (lags) y split temporal

**Qué se hace:** se define la función que convierte la serie horaria en un problema supervisado
tabular (lags 1 a `LOOKBACK` de PM2.5 + variables meteorológicas simultáneas → PM2.5 en `t+1`),
igual que el lookback=24 del paper, y se aplica el split año-independiente.

**Regla importante:** el split se hace **antes** de cualquier imputación o escalado (Sección 1,
regla 2) — acá solo se recorta por fecha, no se toca el contenido todavía.


In [ ]:
def construir_features_ventana(df_ancho, columna_objetivo='PM2.5', lookback=LOOKBACK):
    \"\"\"
    Construye lags 1..lookback de todas las columnas disponibles, más el target
    desplazado 1 paso adelante (pronóstico a 1h, h1, igual que la Tabla 1 del paper).

    Devuelve X (DataFrame de features) e y (Serie del target), alineados por índice,
    ANTES de eliminar filas con NaN (eso se decide después, según la fase).
    \"\"\"
    partes = {}
    for col in df_ancho.columns:
        for lag in range(1, lookback + 1):
            partes[f'{col}_lag{lag}'] = df_ancho[col].shift(lag)

    X = pd.DataFrame(partes, index=df_ancho.index)
    y = df_ancho[columna_objetivo].copy()   # target = valor en el instante t (ya alineado, predicho con lags 1..L)
    return X, y


def dividir_train_test(X, y, fecha_corte):
    en_train = X.index < fecha_corte
    en_test = ~en_train
    return X.loc[en_train], y.loc[en_train], X.loc[en_test], y.loc[en_test]


print('Funciones de ventaneo y split definidas.')

---

## Sección 4: Fase 0 — Persistencia (línea base obligatoria)

**Qué se hace:** se predice PM2.5(t) = PM2.5(t-1), sin ningún modelo. Se evalúa **solo sobre las
horas de test donde el valor real de PM2.5 está observado** (nunca sobre horas imputadas).

**Por qué es obligatoria:** el PM2.5 horario tiene autocorrelación de lag-1 muy alta. Si el RF de
las fases siguientes no supera claramente esta línea base, no está aportando nada por encima de
"copiar la última hora" — es la pregunta que hay que poder responder antes de festejar cualquier R².


In [ ]:
resultados_globales = []  # acá se acumulan las filas de la tabla comparativa final


def evaluar_y_registrar(nombre_fase, y_true, y_pred, n_train=np.nan, n_test=np.nan,
                         tiempo_imputacion=0.0, tiempo_entrenamiento=0.0, tiempo_inferencia=0.0,
                         pct_imputado_train=np.nan):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    resultados_globales.append({
        'fase': nombre_fase,
        'n_train': n_train,
        'n_test': n_test,
        'RMSE': round(rmse, 3),
        'MAE': round(mae, 3),
        'R2': round(r2, 4),
        'tiempo_imputacion_s': round(tiempo_imputacion, 2),
        'tiempo_entrenamiento_s': round(tiempo_entrenamiento, 2),
        'tiempo_inferencia_s': round(tiempo_inferencia, 4),
        'pct_imputado_train': pct_imputado_train,
    })
    print(f'[{nombre_fase}] RMSE={rmse:.3f}  MAE={mae:.3f}  R2={r2:.4f}  '
          f'(n_test={n_test})')
    return rmse, mae, r2

In [ ]:
# Serie real de PM2.5 de FONTIBÓN, recortada al período de test
serie_pm25 = df_ancho_fontibon['PM2.5']
serie_test = serie_pm25.loc[serie_pm25.index >= FECHA_CORTE_TRAIN_TEST]

y_true_persistencia = serie_test
y_pred_persistencia = serie_test.shift(1)   # predicción = valor de la hora anterior

# Solo evaluamos donde AMBOS (real y "predicho") están observados —
# nunca se imputa nada para esta línea base, es intencional.
mascara_valida = y_true_persistencia.notna() & y_pred_persistencia.notna()

t0 = time.perf_counter()
evaluar_y_registrar(
    'Fase 0 - Persistencia',
    y_true_persistencia[mascara_valida],
    y_pred_persistencia[mascara_valida],
    n_test=mascara_valida.sum(),
    tiempo_inferencia=time.perf_counter() - t0,
)

---

## Sección 5: Fase 3 — RF control (sin imputar)

**Qué se hace:** de las variables disponibles en FONTIBÓN, se conservan solo las que tienen
cobertura nativa suficiente (≥ `UMBRAL_COBERTURA_FASE3`) **dentro del período de train** —
las demás se descartan del feature set, no se imputan. Se construyen las ventanas, se eliminan
filas con algún NaN remanente, y se entrena el RF tal cual.

**Rol:** esta fase es el "piso" — mide cuánto rinde el modelo usando exclusivamente datos
100% reales, sin ningún tratamiento estadístico de por medio.


In [ ]:
# Cobertura por variable, calculada SOLO con el período de train (evita fuga)
train_fontibon = df_ancho_fontibon.loc[df_ancho_fontibon.index < FECHA_CORTE_TRAIN_TEST]
cobertura_train = train_fontibon.notna().mean().sort_values(ascending=False)
print('Cobertura por variable en TRAIN (FONTIBÓN):')
print((cobertura_train * 100).round(1))

variables_fase3 = cobertura_train[cobertura_train >= UMBRAL_COBERTURA_FASE3].index.tolist()
if 'PM2.5' not in variables_fase3:
    variables_fase3 = ['PM2.5'] + variables_fase3   # el target siempre debe estar presente

print(f'\nVariables retenidas en Fase 3 (cobertura >= {UMBRAL_COBERTURA_FASE3:.0%}): {variables_fase3}')

In [ ]:
df_fase3 = df_ancho_fontibon[variables_fase3]

X_f3, y_f3 = construir_features_ventana(df_fase3, columna_objetivo='PM2.5', lookback=LOOKBACK)
X_train_f3, y_train_f3, X_test_f3, y_test_f3 = dividir_train_test(X_f3, y_f3, FECHA_CORTE_TRAIN_TEST)

# Sin imputación: se eliminan filas con cualquier NaN (en features o en target)
filas_ok_train = X_train_f3.notna().all(axis=1) & y_train_f3.notna()
filas_ok_test = X_test_f3.notna().all(axis=1) & y_test_f3.notna()

X_train_f3, y_train_f3 = X_train_f3[filas_ok_train], y_train_f3[filas_ok_train]
X_test_f3, y_test_f3 = X_test_f3[filas_ok_test], y_test_f3[filas_ok_test]

print(f'Fase 3 — filas de train utilizables: {len(X_train_f3):,}')
print(f'Fase 3 — filas de test utilizables:  {len(X_test_f3):,}')

In [ ]:
modelo_f3 = RandomForestRegressor(n_estimators=N_ESTIMATORS_RF, random_state=RANDOM_STATE, n_jobs=-1)

t0 = time.perf_counter()
modelo_f3.fit(X_train_f3, y_train_f3)
tiempo_train_f3 = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_f3 = modelo_f3.predict(X_test_f3)
tiempo_inferencia_f3 = time.perf_counter() - t0

evaluar_y_registrar(
    'Fase 3 - RF control (sin imputar)',
    y_test_f3, y_pred_f3,
    n_train=len(X_train_f3), n_test=len(X_test_f3),
    tiempo_imputacion=0.0,
    tiempo_entrenamiento=tiempo_train_f3,
    tiempo_inferencia=tiempo_inferencia_f3,
    pct_imputado_train=0.0,
)

---

## Sección 6: Fase 1 — RF + MissForest (intra-estación)

**Qué se hace:** se imputan **todas** las variables de la réplica (no solo las de cobertura alta)
usando `IterativeImputer` con `RandomForestRegressor` como estimador base — la aproximación estándar
en scikit-learn a MissForest. El imputador se ajusta (`fit`) **solo con el período de train**, y se
aplica (`transform`) tanto a train como a test — así el modelo de imputación nunca ve datos de 2024.

**Advertencia que ya discutimos:** esto imputa también huecos estructurales largos (meses/años),
no solo huecos cortos. Los resultados de esta fase hay que leerlos junto con `pct_imputado_train`
para saber cuánto del set de entrenamiento es señal real vs. inventada por el imputador.


In [ ]:
df_fase1 = df_ancho_fontibon[VARIABLES_REPLICA].copy()

train_f1 = df_fase1.loc[df_fase1.index < FECHA_CORTE_TRAIN_TEST]
test_f1 = df_fase1.loc[df_fase1.index >= FECHA_CORTE_TRAIN_TEST]

pct_faltante_train_f1 = train_f1.isna().mean().mean()
print(f'% de celdas faltantes en TRAIN (Fase 1, todas las variables): {pct_faltante_train_f1:.1%}')

imputador_f1 = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1),
    max_iter=8,
    random_state=RANDOM_STATE,
)

t0 = time.perf_counter()
train_f1_imputado = pd.DataFrame(
    imputador_f1.fit_transform(train_f1), index=train_f1.index, columns=train_f1.columns
)
# IMPORTANTE: transform, no fit_transform, sobre test — el imputador ya está "cerrado" con train
test_f1_imputado = pd.DataFrame(
    imputador_f1.transform(test_f1), index=test_f1.index, columns=test_f1.columns
)
tiempo_imputacion_f1 = time.perf_counter() - t0

print(f'Imputación Fase 1 completa en {tiempo_imputacion_f1:.1f}s')

df_fase1_completo = pd.concat([train_f1_imputado, test_f1_imputado]).sort_index()

In [ ]:
X_f1, y_f1 = construir_features_ventana(df_fase1_completo, columna_objetivo='PM2.5', lookback=LOOKBACK)
X_train_f1, y_train_f1, X_test_f1, y_test_f1 = dividir_train_test(X_f1, y_f1, FECHA_CORTE_TRAIN_TEST)

# Tras imputar ya no debería haber NaN en features, pero por seguridad:
X_train_f1 = X_train_f1.dropna()
y_train_f1 = y_train_f1.loc[X_train_f1.index]

# El TARGET de test se evalúa contra el valor REAL observado, no el imputado (regla 4).
# Por eso se reconstruye y_test_f1 desde la serie ORIGINAL (con NaN), y se filtran esas filas.
y_test_f1_real = df_ancho_fontibon['PM2.5'].loc[X_test_f1.index]
filas_test_reales = y_test_f1_real.notna()

X_test_f1 = X_test_f1[filas_test_reales]
y_test_f1_real = y_test_f1_real[filas_test_reales]

print(f'Fase 1 — filas de train: {len(X_train_f1):,}')
print(f'Fase 1 — filas de test (con PM2.5 real observado): {len(X_test_f1):,}')

In [ ]:
modelo_f1 = RandomForestRegressor(n_estimators=N_ESTIMATORS_RF, random_state=RANDOM_STATE, n_jobs=-1)

t0 = time.perf_counter()
modelo_f1.fit(X_train_f1, y_train_f1)
tiempo_train_f1 = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_f1 = modelo_f1.predict(X_test_f1)
tiempo_inferencia_f1 = time.perf_counter() - t0

evaluar_y_registrar(
    'Fase 1 - RF + MissForest (intra-estación)',
    y_test_f1_real, y_pred_f1,
    n_train=len(X_train_f1), n_test=len(X_test_f1),
    tiempo_imputacion=tiempo_imputacion_f1,
    tiempo_entrenamiento=tiempo_train_f1,
    tiempo_inferencia=tiempo_inferencia_f1,
    pct_imputado_train=round(pct_faltante_train_f1 * 100, 1),
)

---

## Sección 7: Fase 2 — RF + MissForest + combinación de estaciones vía grafo

**Qué se hace:** se construye un grafo de vecindad entre estaciones usando distancia Haversine
(latitud/longitud), se seleccionan las `K` estaciones más cercanas a FONTIBÓN con buena cobertura
de PM2.5, y sus variables (en la misma hora) se agregan como columnas adicionales al `IterativeImputer`
— así el imputador puede usar "lo que estaba pasando en TUNAL a esa hora" para rellenar un hueco de
FONTIBÓN, en vez de depender solo del patrón estacional interno de FONTIBÓN.

**Por qué esto es distinto de la Fase 1:** la Fase 1 solo tiene las propias variables de FONTIBÓN
como predictores de la imputación. Acá el espacio de predictores es mucho más rico, a costa de más
columnas (más tiempo de cómputo) y de la necesidad de justificar la cercanía geográfica.


In [ ]:
def distancia_haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


# Coordenadas de cada estación (una fila por estación, asumiendo coordenada fija en el dataset)
coords_estaciones = (
    df[['nombre_est', 'latitud', 'longitud']]
    .dropna()
    .drop_duplicates(subset='nombre_est')
    .set_index('nombre_est')
)

lat_obj, lon_obj = coords_estaciones.loc[ESTACION_OBJETIVO, ['latitud', 'longitud']]

coords_estaciones['distancia_km'] = distancia_haversine_km(
    lat_obj, lon_obj, coords_estaciones['latitud'], coords_estaciones['longitud']
)

K_VECINOS = 3
vecinos = (
    coords_estaciones.drop(index=ESTACION_OBJETIVO)
    .sort_values('distancia_km')
    .head(K_VECINOS)
)
print(f'Vecinos más cercanos a {ESTACION_OBJETIVO}:')
vecinos[['distancia_km']]

In [ ]:
estaciones_grafo = [ESTACION_OBJETIVO] + vecinos.index.tolist()
df_ancho_grafo = pivotear_ancho(df, estaciones_grafo, FECHA_INICIO, FECHA_FIN_EXCLUSIVA)

print('Forma de la tabla ancha combinada (FONTIBÓN + vecinos):', df_ancho_grafo.shape)
df_ancho_grafo.columns.tolist()[:12]

In [ ]:
train_f2 = df_ancho_grafo.loc[df_ancho_grafo.index < FECHA_CORTE_TRAIN_TEST]
test_f2 = df_ancho_grafo.loc[df_ancho_grafo.index >= FECHA_CORTE_TRAIN_TEST]

# % faltante calculado solo sobre las columnas propias de FONTIBÓN (las de interés real),
# no sobre las columnas de vecinos (que existen solo para ayudar a imputar).
cols_propias = VARIABLES_REPLICA
pct_faltante_train_f2 = train_f2[cols_propias].isna().mean().mean()
print(f'% de celdas faltantes en TRAIN, variables propias de FONTIBÓN: {pct_faltante_train_f2:.1%}')

imputador_f2 = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1),
    max_iter=8,
    random_state=RANDOM_STATE,
)

t0 = time.perf_counter()
train_f2_imputado = pd.DataFrame(
    imputador_f2.fit_transform(train_f2), index=train_f2.index, columns=train_f2.columns
)
test_f2_imputado = pd.DataFrame(
    imputador_f2.transform(test_f2), index=test_f2.index, columns=test_f2.columns
)
tiempo_imputacion_f2 = time.perf_counter() - t0

print(f'Imputación Fase 2 completa en {tiempo_imputacion_f2:.1f}s '
      f'(vs {tiempo_imputacion_f1:.1f}s en Fase 1 — más columnas, más costo)')

# Nos quedamos solo con las columnas propias de FONTIBÓN para el modelo de pronóstico;
# las columnas de vecinos cumplieron su función (ayudar a imputar) y no siguen como features
# del RF de pronóstico, para mantener el mismo espacio de features que las Fases 1 y 3.
df_fase2_completo = pd.concat([
    train_f2_imputado[cols_propias], test_f2_imputado[cols_propias]
]).sort_index()

In [ ]:
X_f2, y_f2 = construir_features_ventana(df_fase2_completo, columna_objetivo='PM2.5', lookback=LOOKBACK)
X_train_f2, y_train_f2, X_test_f2, y_test_f2 = dividir_train_test(X_f2, y_f2, FECHA_CORTE_TRAIN_TEST)

X_train_f2 = X_train_f2.dropna()
y_train_f2 = y_train_f2.loc[X_train_f2.index]

y_test_f2_real = df_ancho_fontibon['PM2.5'].loc[X_test_f2.index]
filas_test_reales_f2 = y_test_f2_real.notna()

X_test_f2 = X_test_f2[filas_test_reales_f2]
y_test_f2_real = y_test_f2_real[filas_test_reales_f2]

print(f'Fase 2 — filas de train: {len(X_train_f2):,}')
print(f'Fase 2 — filas de test (con PM2.5 real observado): {len(X_test_f2):,}')

In [ ]:
modelo_f2 = RandomForestRegressor(n_estimators=N_ESTIMATORS_RF, random_state=RANDOM_STATE, n_jobs=-1)

t0 = time.perf_counter()
modelo_f2.fit(X_train_f2, y_train_f2)
tiempo_train_f2 = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_f2 = modelo_f2.predict(X_test_f2)
tiempo_inferencia_f2 = time.perf_counter() - t0

evaluar_y_registrar(
    f'Fase 2 - RF + MissForest + grafo (K={K_VECINOS})',
    y_test_f2_real, y_pred_f2,
    n_train=len(X_train_f2), n_test=len(X_test_f2),
    tiempo_imputacion=tiempo_imputacion_f2,
    tiempo_entrenamiento=tiempo_train_f2,
    tiempo_inferencia=tiempo_inferencia_f2,
    pct_imputado_train=round(pct_faltante_train_f2 * 100, 1),
)

---

## Sección 8: Tabla comparativa y conclusiones

**Qué se hace:** se consolidan las 4 filas (Persistencia + 3 fases) en una sola tabla, y se
grafican RMSE/R² junto con el costo de tiempo de cada fase, para poder leer de un vistazo el
trade-off precisión vs. costo computacional vs. cantidad de dato real utilizado.


In [ ]:
tabla_resultados = pd.DataFrame(resultados_globales)
tabla_resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.barplot(data=tabla_resultados, x='fase', y='RMSE', ax=axes[0], color='#4C72B0')
axes[0].set_title('RMSE por fase (menor es mejor)')
axes[0].tick_params(axis='x', rotation=35)
for lbl in axes[0].get_xticklabels():
    lbl.set_ha('right')

sns.barplot(data=tabla_resultados, x='fase', y='R2', ax=axes[1], color='#55A868')
axes[1].set_title('R² por fase (mayor es mejor)')
axes[1].tick_params(axis='x', rotation=35)
for lbl in axes[1].get_xticklabels():
    lbl.set_ha('right')

tiempos_totales = (
    tabla_resultados['tiempo_imputacion_s']
    + tabla_resultados['tiempo_entrenamiento_s']
    + tabla_resultados['tiempo_inferencia_s']
)
axes[2].bar(tabla_resultados['fase'], tiempos_totales, color='#C44E52')
axes[2].set_title('Tiempo total (imputación + train + inferencia), s')
axes[2].tick_params(axis='x', rotation=35)
for lbl in axes[2].get_xticklabels():
    lbl.set_ha('right')

plt.tight_layout()
plt.show()

In [ ]:
# Comparación de R² contra los números reportados en el paper (Tabla 1, h1)
# — el R² es la métrica que elegimos como escala-libre para comparar entre datasets.
comparacion_paper = pd.DataFrame({
    'fuente': ['Beijing (paper) - STN', 'Taizhou (paper) - STN'] + tabla_resultados['fase'].tolist(),
    'R2': [0.937, 0.924] + tabla_resultados['R2'].tolist(),
})
comparacion_paper.sort_values('R2', ascending=False)

### Cómo leer esta tabla

- Si la **Fase 0 (persistencia)** ya tiene un R² alto (ver pregunta 8 de la guía teórica), cualquier
  fase de RF debe superarla claramente para justificar el modelo — si no la supera por un margen
  amplio, el "aprendizaje" del RF es marginal frente a simplemente copiar la hora anterior.
- Comparar **Fase 3 vs Fase 1**: si el R² sube pero `pct_imputado_train` es alto, hay que sospechar
  del resultado — parte de esa mejora puede venir de que el imputador "inventó" una señal fácil de
  aprender, no de una mejora real en la capacidad predictiva.
- Comparar **Fase 1 vs Fase 2**: si el R² sube y el tiempo de imputación también sube mucho, ahí está
  el trade-off precisión-costo que hay que documentar explícitamente en la conclusión del proyecto.
- La comparación contra Beijing/Taizhou es orientativa (dominios y regímenes de contaminación
  distintos) — sirve para decir "estamos en el mismo orden de magnitud de R²", no para afirmar
  superioridad o inferioridad directa.


---

## Notas y próximos pasos

- Este cuaderno resuelve el pronóstico a **1 hora (h1)**, igual que la Tabla 1 del paper. Para
  replicar las Tablas 2-5 (multi-paso, 6/12/24/48h) hay que extender `construir_features_ventana`
  para generar targets a múltiples horizontes, y decidir entre pronóstico autorregresivo o
  multisalida directa — evitando el atajo de "modelos independientes por intervalo" que señalamos
  como injusto en la pregunta 12 de la guía teórica.
- La validación de hiperparámetros (si se decide ajustar `N_ESTIMATORS_RF` u otros) debe hacerse con
  **ventanas deslizantes dentro del período de train**, nunca con una partición aleatoria del 10%
  (pregunta 7 de la guía teórica) — este cuaderno no la incluye todavía porque usa los hiperparámetros
  por defecto razonables de RF, pero es el siguiente paso antes de pasar a LSTM/STN.
- Cuando se pase a LSTM/STN, el escalador (MinMax o estándar) debe ajustarse únicamente con
  `X_train`/`y_train` de la fase elegida, nunca con el dataset completo (pregunta 5 de la guía teórica).
